# Strawberry RUL Lab Overview

This is the strawberry-only lab for improving RUL prediction. It reads the consolidated metadata at `data/02_processed/strawberry/final/metadata.csv`, infers fruit-level splits from `data/03_split/strawberry`, and ignores avocado data.


In [ ]:
from pathlib import Path
import sys

LAB_DIR = Path.cwd()
if LAB_DIR.name != 'strawberry':
    LAB_DIR = Path('notebooks/strawberry').resolve()
PROJECT_ROOT = LAB_DIR.parents[1]
sys.path.insert(0, str(LAB_DIR))

import lab_utils as lab

print('Project root:', PROJECT_ROOT)
print('Metadata:', lab.METADATA_PATH)
print('Split root:', lab.SPLIT_ROOT)


In [ ]:
lab.selected_training_config()


In [ ]:
counts = lab.sequence_counts([3, 5, 8, 10, 12])
counts.pivot_table(index=['split', 'fruit_id'], columns='seq_len', values='sequences', aggfunc='sum')


## Search Strategy

1. Audit strawberry metadata, sensor provenance, target distribution, and leakage-safe sequence counts.
2. Run the fast LOOCV lab sweep to compare sequence length and fusion mode.
3. Push the best evidence-backed setting into `configs/strawberry_training.json`.
4. Train A/B/C/D with the shared LOOCV package.


In [ ]:
selected = lab.selected_training_config()
{
    'model_keys': selected['model_keys'],
    'seq_len': selected['seq_len'],
    'fusion_mode': selected['fusion_mode'],
    'temporal_pooling': selected['temporal_pooling'],
    'loss': 'SmoothL1Loss',
    'regularization': ['light augmentation', 'dropout=0.25', 'ReduceLROnPlateau', 'early stopping'],
}
